**bold text**# 📘 Lecture Handout Generator (Audio + Image)

This notebook converts lecture audio and board images into structured, printable handouts using the following pipeline:

1. **ASR (Whisper)** – Transcribes lecture audio.  
2. **Regex Cleanup** – Fixes common transcription issues.  
3. **LLM (Gemini Flash 2.5)** – Generates a structured handout using both transcript and image.  
4. **Markdown to PDF** – Converts output to a clean PDF.

> The system preserves the lecture’s original language and terminology, with **special support for Persian lectures** including proper handling of Persian text and visuals.

API Key: [https://aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)

#1_ ASR(Whisper)

To convert lecture audio into text, this system uses Whisper, a local, open-source automatic speech recognition (ASR) model developed by OpenAI.


GitHub Repository: https://github.com/openai/whisper

In [ ]:
# Install whisper requirmnet
!pip install -U openai-whisper
!pip install git+https://github.com/openai/whisper.git
!pip install --upgrade --no-deps --force-reinstall git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg
!pip install setuptools-rust

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-y_tbpg1_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-y_tbpg1_
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-7eg0ey4h
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-7eg0ey4h
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=8548b424b897dbc7a4feffe02

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.m4a to test.m4a


In [ ]:
import whisper
model = whisper.load_model("turbo")

In [ ]:
result = model.transcribe("test.m4a" ,language="fa")
print(result["text"])

 موسیقی موسیقی خب چند تا روش گفتیم برای سخت این جدوانه و اینها الاروان از همه چی بود؟ بس در از بس گفتیم یه جاهایی بیه جاهایی با همین استداروان میشه غزیه رو حل کرد فقط به کانفیلیت که رسیدیم چگاه کنیم؟ با اولویت یه چیزی غزیه رو رفت روشی کنیم به به کارش دیگه سراب الاروان و الاروان و همه این همه هم منیم که یه مثال شما دیدید با اولویت بزینیم و اولویت و شرکت نسیم با اولویت خانه های خالی خانه های خالی یعنی اینکه اگر یدول شما برد توی حالت خانه های خالی شما باید چی گذرش بدید؟ ایرارو بگذرش ایرارو که اولین بحثه گذرش میدیم یه ما تویی خط خودشی داره؟ اخوانه داره؟ باید استعبشه باید استعبشه همه دو تا بحثه دیگه باشید داره یه هایی میتونیم توییم بزارش کمک کنیم به معنامه نویست برای اینکه مشتله همه شما باید کنیم یا نه فقط بگیم ارارو فقط بگیم ایرارو داری خودت رو رفت کن یا بتونیم چگاه کنیم؟ یکه پیغام اجودیت پیشتر بدیم که بتونیم بحث دومونم اونه که اگر به ایرارو خوردیم متوقف شدیم تازمان اینکه ارارو رفتش شد ادامه ندیم کار رو کنیم یا نه بقیه این کدر رو از دقیقه چگاه کنیم؟ بررسی کنیم که اتفاهم خب جواب این دو دو س

#2_ RegEx (Text Cleaning)

The output from Whisper often contains various errors — such as duplicate words or characters, and even letters from other languages.
We can handle these issues using regular expressions with Python’s `re` module.

In [ ]:
import re

def remove_duplicate_words(input):
    # Regex to matching repeated words
    regex = r'\b(\w+)(?:\W+\1\b)+'
    return re.sub(regex, r'\1', input, flags=re.IGNORECASE)

def handle_duplicates_alphabet(text):
    """Normalizes repeated characters in words based on language rules.

    Rules:
        - Persian/Arabic: Removes ALL consecutive duplicates (سلامم → سلام)
        - English: Allows max 2 repeats (Heeelllo → Heelloo)
        - Numbers: Unchanged

    Args:
        text (str): Input text with mixed languages/numbers.

    Returns:
        str: Text with processed words.

    """

    def process_word(word):
        # Check if the word is a number (do nothing)
        if word.isdigit():
            return word

        # Check if the word is Persian (delete all duplicates)
        if re.search(r'[\u0600-\u06FF]', word):  # Persian/Arabic Unicode range
            processed = re.sub(r'(.)\1+', r'\1', word)
            return processed

        # For English words (keep max 2 duplicates, remove 3+)
        processed = re.sub(r'(.)\1{2,}', r'\1\1', word)
        return processed

    # Split into words and process each one
    tokens = re.findall(r'(\s+|\d+|\w+|[^\w\s])', text)
    return ''.join([process_word(token) for token in tokens])


def filter_allowed_chars(text):
    """Filters text to allow only Persian/English letters, numbers, and common punctuation.

    Allowed:
        - Persian: \u0600-\u06FF + پ (067E), چ (0686), etc.
        - English: A-Za-z
        - Numbers: 0-9 and Persian digits (\u0660-\u0669)
        - Punctuation: English (!@#$) + Persian (،؛؟)

    Args:
        text (str): Raw input text with possible invalid characters.

    Returns:
        str: Sanitized text with disallowed characters removed.

    """

    allowed_pattern = re.compile(
        r'['
        r'\u0600-\u06FF\u067E\u0686\u06AF\u0698' # Persian
        r'A-Za-z'  # English
        r'0-9\u0660-\u0669'  # \number
        r'\s'  # space
        r'!@#\$%\^&\*\(\)\-_=\+\[\]\{\};:\'",<>\.\/?\\|~`'  # Enlish puctuation marks
        r'،؛؟«»'  # Persian puctuation marks
        r']',
        flags=re.UNICODE
    )

    cleaned_text = ''.join(allowed_pattern.findall(text))
    return cleaned_text

In [ ]:
s1 = handle_duplicates_alphabet(result["text"])
s2 = remove_duplicate_words(s1)
s3 = filter_allowed_chars(s2)
print(len(result["text"]),len(s1),len(s2),len(s3))
print(s3)

41304 41148 40595 40581
 موسیقی خب چند تا روش گفتیم برای سخت این جدوانه و اینها الاروان از همه چی بود؟ بس در از بس گفتیم یه جاهای بیه جاهای با همین استداروان میشه غزیه رو حل کرد فقط به کانفیلیت که رسیدیم چگاه کنیم؟ با اولویت یه چیزی غزیه رو رفت روشی کنیم به کارش دیگه سراب الاروان و الاروان و همه این همه هم منیم که یه مثال شما دیدید با اولویت بزینیم و اولویت و شرکت نسیم با اولویت خانه های خالی خانه های خالی یعنی اینکه اگر یدول شما برد توی حالت خانه های خالی شما باید چی گذرش بدید؟ ایرارو بگذرش ایرارو که اولین بحثه گذرش میدیم یه ما توی خط خودشی داره؟ اخوانه داره؟ باید استعبشه باید استعبشه همه دو تا بحثه دیگه باشید داره یه های میتونیم تویم بزارش کمک کنیم به معنامه نویست برای اینکه مشتله همه شما باید کنیم یا نه فقط بگیم ارارو فقط بگیم ایرارو داری خودت رو رفت کن یا بتونیم چگاه کنیم؟ یکه پیغام اجودیت پیشتر بدیم که بتونیم بحث دومونم اونه که اگر به ایرارو خوردیم متوقف شدیم تازمان اینکه ارارو رفتش شد ادامه ندیم کار رو کنیم یا نه بقیه این کدر رو از دقیقه چگاه کنیم؟ برسی کنیم که اتفاهم خب جواب این


#3_ LLM (Gemini Flash 2.5)

To generate a structured handout from the voice transcription and board image, we use the Gemini Flash 2.5 API.

You can get your own free API key from Google AI Studio:
 https://aistudio.google.com/app/apikey

In [ ]:
from google import genai
from google.genai import types
client = genai.Client(api_key="API key")

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test1_1.png to test1_1.png
Saving test1_2.png to test1_2.png


In [ ]:
# Upload the first image
image1_path = "test1_1.png"
image1 = client.files.upload(file=image1_path)
# Upload the second image
image2_path = "test1_2.png"
image2 = client.files.upload(file=image1_path)
# you can add more jsut like this

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(

        system_instruction="""You are a helpful assistant that generates structured, ready-to-use handouts from two inputs:
1. Images of a course board (e.g., whiteboard, blackboard, or presentation slides).
2. Transcribed speech of the teacher’s lecture (in the original language used by the teacher).

Output Requirements:
Structured Format: Organize content into clear sections (e.g., Topic, Key Concepts, Definitions, Examples, Steps/Procedures, Summary).
Language Consistency: Use the teacher’s exact language and terminology. If the lecture is in Persian, the handout must also be in Persian.
Visual Integration: Include diagrams/equations from the images with labeled references (e.g., "Fig. 1: [description]").
No Omissions: Ensure no part of the lecture content is missed.
Formatting Rules:
Start only with the main topic header (do not include metadata or explanations before it).
Use correct Markdown formatting (e.g., #, ##, -) for hierarchy and clarity."""),

    contents=[s3,
              image1,
              image2
              ])

print(response.text)

# الگوریتم‌های تجزیه (Parsing Algorithms)

## مدیریت خطا در فرآیند تجزیه (Error Management in Parsing)
هنگامی که یک پارسر با خطا مواجه می‌شود، دو رویکرد اصلی برای مدیریت آن وجود دارد:

### 1. قواعد تولید خطا (Error Productions)
*   **ایده:** قواعد جدیدی به گرامر اضافه می‌کنیم که الگوهای خطای رایج را بپذیرند تا پارسر بتواند با آن‌ها ادامه دهد.
*   **مثال:** اضافه کردن `E -> E + . E` به گرامر برای پذیرش `+ .` به عنوان یک ساختار (که در حالت عادی خطا است).
*   **مزایا:** امکان ادامه کار پارسر و تلاش برای بازیابی از خطا.
*   **معایب:**
    *   **ابهام (Ambiguity):** اضافه کردن این قواعد می‌تواند گرامر را مبهم کند، به این معنی که یک رشته می‌تواند چندین درخت تجزیه داشته باشد.
    *   **پیچیدگی:** گرامر بسیار بزرگ و پیچیده می‌شود.
    *   **گزارش خطا:** گزارش خطاها کمتر دقیق می‌شود، زیرا سیستم سعی می‌کند خطا را "اصلاح" کند تا ادامه دهد.
    *   **پوشش محدود:** نمی‌توان همه خطاهای ممکن را با قواعد خطا پوشش داد.

### 2. حالت وحشت (Panic Mode)
*   **ایده:** هنگامی که پارسر با یک خطا مواجه می‌شود،


# 4_ Generate PDF File

To improve readability and make the output more flexible for future use, we convert the LLM's response into a PDF file.
Since the LLM already returns the result in Markdown format, we use the `markdown_pdf` package to create the PDF.

In [ ]:
!pip install markdown_pdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 95.2 MB/s eta 0:00:00


In [ ]:
from markdown_pdf import MarkdownPdf, Section

# Initialize PDF with RTL settings
pdf = MarkdownPdf(toc_level=2)
content = f"""
<style>
  html, body {{
    direction: rtl;
    font-family: "Vazirmatn", sans-serif;
    text-align: right;
  }}
</style>
"""+response.text

pdf.add_section(Section(content))
pdf.save("test.pdf")

In [ ]:
# Download the file
files.download('test.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>